# 2) Eddy FKE preprocessing

**Input: .zarr file with model daily FKE and daily eddy outlines (py-eddy-tracker .nc output from 1_ notebook)**

**Output: eddy outline data in a workable .zarr file**



In this notebook you prepare the eddy dataset to be used to extract the FKE in the next notebooks



In [ ]:
import numpy as np
import xarray as xr
import dask
import dask.array as da
from dask.distributed import Client, LocalCluster
from matplotlib.path import Path
import os

**Here you read in your .zarr fke file**

In [3]:
eke_path = "/work/bk1377/b382615/eddies/"
model = "ICON"
icon_eke = xr.open_zarr(eke_path + "ICON_hist_GS.zarr", chunks={})

icon_eke

FileNotFoundError: /work/bk1377/b382615/eddies/NEMO_hist_GS.zarr does not exist

**Because the model output is slightly different  always, you have to pre-process some data to make it work**

In [9]:
track_path = "/work/bk1377/b382618/model_eddy_eval/historical/nemo/eddyTracks/" ## where your eddies are
sorted_zarr = "/scratch/b/b382618/eeke/" + "consolidated_tracks_sorted_"+model+"_mid.zarr" ## where you want the mid-file file to be stored

In [9]:
lat_1d = icon_eke['lat'].values
lon_1d = icon_eke['lon'].values

In [6]:
## Consolidate, Sort, & Save Dataset
## we read in the short tracks and the "normal" tracks and merge all of them together in one xarray

EDDY_TYPE_MAP = {
    'anticyclonic_long':  0,
    'cyclonic_long':      1,
    'anticyclonic_short': 2,
    'cyclonic_short':     3,
}

if not os.path.exists(sorted_zarr):
    vars_to_keep = ['time', 'latitude', 'longitude', 'track',
                    'effective_contour_latitude', 'effective_contour_longitude']

    a_icon       = xr.open_dataset(track_path+"/anticyclonic_tracks.nc",   chunks={})[vars_to_keep]
    c_icon       = xr.open_dataset(track_path+"/cyclonic_tracks.nc",       chunks={})[vars_to_keep]
    a_icon_short = xr.open_dataset(track_path+"/anticyclonic_short.nc",    chunks={})[vars_to_keep]
    c_icon_short = xr.open_dataset(track_path+"/cyclonic_short.nc",        chunks={})[vars_to_keep]

    for ds, name in zip(
        [a_icon,           c_icon,          a_icon_short,       c_icon_short],
        ['anticyclonic_long','cyclonic_long','anticyclonic_short','cyclonic_short']
    ):
        ds.coords['eddy_type'] = np.int8(EDDY_TYPE_MAP[name])

    combined_ds = xr.concat(
        [a_icon, c_icon, a_icon_short, c_icon_short],
        dim='obs', data_vars='minimal'
    )#.chunk({'obs': 100_000})

    # Sort by time
    time_vals   = combined_ds['time'].values
    sort_idx    = np.argsort(time_vals, kind='stable')
    
    def _sort_ds(ds, idx):
        data_vars = {v: (ds[v].dims, ds[v].values[idx]) for v in ds.data_vars}
        coords    = {c: (ds[c].dims, ds[c].values[idx])
                    for c in ds.coords if 'obs' in ds[c].dims}
        return xr.Dataset(data_vars, coords=coords)

    combined_sorted = _sort_ds(combined_ds, sort_idx)  # Bit of a hack to avoid making a big dask graph when sorting...
    #combined_sorted = combined_ds.isel(obs=sort_idx)
    time_sorted = time_vals[sort_idx]

    # Vectorised cumcount-within-group: obs_day = 0-based position within each unique timestamp
    changes = np.concatenate([
        [0],
        np.where(np.diff(time_sorted.view('i8')) != 0)[0] + 1,
        [len(time_sorted)]
    ])
    obs_day = (np.arange(len(time_sorted))
               - changes[np.searchsorted(changes, np.arange(len(time_sorted)), side='right') - 1])

    combined_sorted = combined_sorted.assign_coords(
        obs_day=('obs', obs_day.astype(np.uint16)),  # 0-based index of this eddy within its day
    ).chunk({'obs': 100_000})

    out_ds = combined_sorted.copy()
    for name in list(out_ds.data_vars) + list(out_ds.coords):
        out_ds[name].encoding = {}
    out_ds.to_zarr(sorted_zarr, mode='w')

else:
    combined_sorted = xr.open_zarr(sorted_zarr, chunks={'obs': 100_000})
    time_sorted = combined_sorted['time'].values

# Reconstruct day-boundary index array
changes = np.concatenate([
    [0],
    np.where(np.diff(time_sorted.view('i8')) != 0)[0] + 1,
    [len(time_sorted)]
])
n_days       = len(changes) - 1
unique_times = time_sorted[changes[:-1]]

# Fix time rounding error
combined_sorted['time'] = combined_sorted['time'].dt.round('s')

combined_sorted

<xarray.Dataset> Size: 46GB
Dimensions:                      (obs: 55706949, NbSample: 50)
Coordinates:
    eddy_type                    (obs) int8 56MB dask.array<chunksize=(100000,), meta=np.ndarray>
    obs_day                      (obs) uint16 111MB dask.array<chunksize=(100000,), meta=np.ndarray>
Dimensions without coordinates: obs, NbSample
Data variables:
    effective_contour_latitude   (obs, NbSample) float64 22GB dask.array<chunksize=(100000, 50), meta=np.ndarray>
    effective_contour_longitude  (obs, NbSample) float64 22GB dask.array<chunksize=(100000, 50), meta=np.ndarray>
    latitude                     (obs) float32 223MB dask.array<chunksize=(100000,), meta=np.ndarray>
    longitude                    (obs) float32 223MB dask.array<chunksize=(100000,), meta=np.ndarray>
    time                         (obs) datetime64[ns] 446MB dask.array<chunksize=(100000,), meta=np.ndarray>
    track                        (obs) uint32 223MB dask.array<chunksize=(100000,), meta=np.ndarray>

In [7]:
cluster = LocalCluster(n_workers=64, threads_per_worker=1)
client  = Client(cluster)
print(client.dashboard_link)

/home/b/b382618/.conda/envs/eddy_env/lib/python3.11/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 41087 instead
  warnings.warn(


http://127.0.0.1:41087/status


In [8]:
## Reindex & Reshape into (time, obs_day) data

# reindexed_zarr = "/scratch/b/b382618/eeke/" + "consolidated_tracks_reindexed_2"+model+".zarr"
reindexed_zarr = "/scratch/b/b382618/eeke/" + "consolidated_tracks_reindexed_"+model+"_final.zarr" #this is the actual end file we want

if not os.path.exists(reindexed_zarr):
    ds_idx = combined_sorted.set_coords("time")

    # Build (time, obs_day) index on obs, then unstack to 2D
    combined_obsday_time = (
        ds_idx
        .set_index(obs=["time", "obs_day"])
        .unstack("obs")
        .transpose("obs_day", "time", ...)
    ).chunk({'obs_day': -1, 'time': 1, 'NbSample': -1})
    combined_obsday_time.to_zarr(reindexed_zarr, mode='w')
    
else:
    combined_obsday_time = xr.open_zarr(reindexed_zarr, chunks={'obs_day': -1, 'time': 1, 'NbSample': -1})

combined_obsday_time

<xarray.Dataset> Size: 48GB
Dimensions:                      (obs_day: 7252, time: 8035, NbSample: 50)
Coordinates:
  * obs_day                      (obs_day) uint16 15kB 0 1 2 ... 7249 7250 7251
  * time                         (time) datetime64[ns] 64kB 1993-01-01T12:00:...
    eddy_type                    (obs_day, time) float32 233MB dask.array<chunksize=(7252, 1), meta=np.ndarray>
Dimensions without coordinates: NbSample
Data variables:
    effective_contour_latitude   (obs_day, time, NbSample) float64 23GB dask.array<chunksize=(7252, 1, 50), meta=np.ndarray>
    effective_contour_longitude  (obs_day, time, NbSample) float64 23GB dask.array<chunksize=(7252, 1, 50), meta=np.ndarray>
    latitude                     (obs_day, time) float32 233MB dask.array<chunksize=(7252, 1), meta=np.ndarray>
    longitude                    (obs_day, time) float32 233MB dask.array<chunksize=(7252, 1), meta=np.ndarray>
    track                        (obs_day, time) float64 466MB dask.array<chunksize=(7252, 1), meta=np.ndarray>

**This output file is now used for applying to FKE maps and calculating eddy intensity and decomposition of FKE**